# Buka | All Python figures + all statistical/Excel tables\n\n**Yuklash:** faqat `BOKA_COLAB_INPUTS_ONLY_20261109.zip` fayli.\n\n**Ishga tushirish:** Runtime → Run all.

In [ ]:
# -*- coding: utf-8 -*-
"""
BO'KA (12 April 2026) -- ALL PYTHON FIGURES / ALL TABLES / ONE-UPLOAD PIPELINE
Run in a SINGLE Google Colab code cell or as a local Python program.
INPUT: BOKA_COLAB_INPUTS_ONLY_20261109.zip
Independent reconstruction from 155 Optosky TXT + Sentinel GEE source 149 CSV
+ official SRF (CSV) + author-confirmed 144-row GlobalID crosswalk.

'Optosky reflectance = C4/C3
import sys, subprocess, io, os, re, json, zipfile, hashlib, warnings
from pathlib import Path

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.integrate import trapezoid
    from scipy.stats import pearsonr, spearmanr
    from scipy.spatial import cKDTree
    from pyproj import Transformer
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','-q','install',
                           'numpy','pandas','scipy','matplotlib','pyproj'])
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.integrate import trapezoid
    from scipy.stats import pearsonr, spearmanr
    from scipy.spatial import cKDTree
    from pyproj import Transformer

# ================================================================
# 1. SOURCE PACKAGE (UPLOAD ONCE IN COLAB)
# ================================================================
PACKAGE_NAME = 'BOKA_COLAB_INPUTS_ONLY_20261109.zip'
OUT = Path(os.environ.get('BOKA_OUTPUT_DIR', '/content/BOKA_REPRODUCED_RESULTS' if Path('/content').is_dir()
           else str(Path.cwd()/'BOKA_REPRODUCED_RESULTS')))
OUT.mkdir(parents=True,exist_ok=True)
BOOTSTRAP_REPEATS = int(os.getenv('BOKA_BOOTSTRAP_REPEATS', '600'))  # use 2000 for expanded uncertainty analysis
MORAN_PERMUTATIONS = int(os.getenv('BOKA_MORAN_PERMUTATIONS', '999'))
RANDOM_SEED = 20261109
INDEXES = ('NDVI','GNDVI','NDRE')
BANDS = ('B3','B4','B5','B8','B8A')
SOIL_IDS = {'4.1','4.2','4.3','4.4'}
AUXILIARY_NAMES = {'cotton',"sariq o't",'shapaloq','terak','yalpiz'}

package = Path(PACKAGE_NAME)
if not package.exists():
    candidates = list(Path('/mnt/data').glob(PACKAGE_NAME)) if Path('/mnt/data').exists() else []
    if candidates:
        package = candidates[0]
    elif Path('/content').is_dir():
        print('UPLOAD THE FINAL DATA PACKAGE ZIP: ',PACKAGE_NAME)
        from google.colab import files
        incoming=files.upload()
        package = Path(next((name for name in incoming if name.endswith('.zip')), ''))
    else:
        raise FileNotFoundError(PACKAGE_NAME)
assert package.is_file(), 'Input package ZIP not found'


# ================================================================
# 2. FONT CONFIGURATION -- NO FONT UPLOAD PROMPT
# ================================================================
# If Times New Roman is available, use it. Standard Colab runtimes do not
# contain this Microsoft font. In that case install/use Tinos automatically:
# Tinos is an open-source, metrically compatible Times-style serif font,
# but it is NOT the proprietary Times New Roman family.
from matplotlib import font_manager as fm

def configure_times_style():
    def font_available(name):
        try:
            path = fm.findfont(fm.FontProperties(family=name), fallback_to_default=False)
            return bool(path and Path(path).is_file())
        except (ValueError, OSError):
            return False

    if font_available('Times New Roman'):
        family='Times New Roman'
    else:
        if not font_available('Tinos'):
            # Ubuntu/Google Colab: free Times-compatible fonts, no uploads.
            if Path('/content').is_dir():
                cmd=['apt-get','update','-qq']
                subprocess.run(cmd,check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
                subprocess.run(['apt-get','install','-y','-qq','fonts-croscore'],
                               check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
            for folder in ('/usr/share/fonts/truetype/croscore',
                           '/usr/share/fonts/truetype/croscore-extra',
                           '/usr/local/share/fonts'):
                for file in Path(folder).glob('Tinos*.ttf'):
                    try: fm.fontManager.addfont(str(file))
                    except (OSError,RuntimeError): pass
        if not font_available('Tinos'):
            raise RuntimeError('Times New Roman and Tinos not found. Cannot guarantee a Times-style output.')
        family='Tinos'
        print('FONT NOTICE: Times New Roman is unavailable in this runtime;')
        print('Tinos (free metric-compatible Times-style font) will be used automatically.')
    plt.rcParams.update({
        'font.family':family, 'font.serif':[family],
        'font.size':10, 'axes.labelsize':11, 'legend.fontsize':9,
        'xtick.labelsize':9, 'ytick.labelsize':9,
        'mathtext.fontset':'stix','pdf.fonttype':42,'ps.fonttype':42,
        'axes.unicode_minus':False,'savefig.facecolor':'white',
        'savefig.dpi':600,'figure.dpi':120,
    })
    print('All graph labels font:',family)
    return family

FIGURE_FONT_FAMILY=configure_times_style()
FIGURE_DPI=600

with zipfile.ZipFile(package) as pack:
    def read_bytes(part):
        matches=[n for n in pack.namelist() if n.endswith(part)]
        assert len(matches)==1,(part,matches)
        return pack.read(matches[0])
    raw_zip_bytes=read_bytes('Optosky_raw_155_TXT.zip')
    srf_bytes=read_bytes('Sentinel2B_SRF_v4_required_bands.csv')
    gee_bytes=read_bytes('GEE_point_values_149.csv')
    crosswalk_bytes=read_bytes('Selected_144_GlobalID_crosswalk.csv')
    # Reference table is used ONLY for checking, NEVER for computing metrics.
    reference_bytes=read_bytes('01_FINAL_PAIRED_144_S2B_VS_OPTOSKY_NDVI_GNDVI_NDRE.csv')

provenance={
 'Optosky_raw_155_TXT.zip':raw_zip_bytes,
 'Sentinel2B_SRF_v4_required_bands.csv':srf_bytes,
 'GEE_original_149.csv':gee_bytes,
 'Confirmed_GlobalID_crosswalk.csv':crosswalk_bytes,
}
pd.DataFrame([{'input':k,'sha256':hashlib.sha256(v).hexdigest(),
               'size_bytes':len(v)} for k,v in provenance.items()]).to_csv(
               OUT/'00_SOURCE_CHECKSUMS.csv',index=False)

def norm_gid(x): return str(x).strip().strip('{}').upper()
def index_ratio(a,b):
    return float((a-b)/(a+b)) if np.isfinite(a+b) and abs(a+b)>1e-14 else np.nan

# ================================================================
# 2. RECREATE ALL OPTOSKY SPECTRA AND SRF CONVOLUTION FROM TXT
# ================================================================
srf=pd.read_csv(io.BytesIO(srf_bytes))
assert {'wavelength_nm',*BANDS}.issubset(srf.columns)
srf=srf.sort_values('wavelength_nm')
srf_wl=srf.wavelength_nm.to_numpy(float)
for b in BANDS: assert np.isfinite(srf[b]).all()
qc_rows=[];opt_rows=[];saved_curves={}
with zipfile.ZipFile(io.BytesIO(raw_zip_bytes)) as z:
    txt_names=sorted(n for n in z.namelist() if n.lower().endswith('.txt'))
    assert len(txt_names)==155, f'Expected 155 TXT, got {len(txt_names)}'
    for filename in txt_names:
        base=Path(filename).stem
        low=base.lower()
        role=('auxiliary' if low in AUXILIARY_NAMES else
              'separate_t_soil' if low.endswith('t') else
              'soil' if base in SOIL_IDS else 'wheat')
        sample=base[:-1] if low.endswith(('b','t')) else base
        if sample in {'6.5b','6.7b'}: sample=sample[:-1]
        a=np.loadtxt(io.BytesIO(z.read(filename)),delimiter='\t')
        assert a.ndim==2 and a.shape[1]>=4,filename
        a=a[np.argsort(a[:,0])]
        wl=a[:,0].astype(float)
        # Author-confirmed format-specific export: C4 / C3
        reflectance=np.divide(a[:,3],a[:,2],
                 out=np.full(len(a),np.nan),where=np.isfinite(a[:,2])&(a[:,2]!=0))
        core=(wl>=450)&(wl<=900)
        valid=core&np.isfinite(reflectance)
        spread=float(np.std(reflectance[valid],ddof=1)) if valid.sum()>1 else np.nan
        flag_high=int(np.sum(valid&(reflectance>1.5)))
        flag_negative=int(np.sum(valid&(reflectance<0)))
        flag_missing=int(np.sum(core&~np.isfinite(reflectance)))
        qc_rows.append(dict(filename=Path(filename).name,sample_id=sample,role=role,
                            n_wavelengths=len(wl),valid_core=int(valid.sum()),
                            above_1p5=flag_high,negative=flag_negative,
                            missing_core=flag_missing,spectral_SD=spread,
                            near_flat=bool(np.isfinite(spread) and spread<0.025)))
        if sample in {'2.10','3.7','3.9','5.5','3.23','4.2','8.6','8.7','7.7'}:
            saved_curves[sample]=(wl,reflectance)
        if role in ('auxiliary','separate_t_soil'):
            continue
        row={'sample_id':sample,'source_file':Path(filename).name,'role':role,
             'QC_above1p5':flag_high,'QC_negative':flag_negative,
             'QC_missing':flag_missing,'QC_near_flat':bool(spread<0.025)}
        for band in BANDS:
            weight=np.interp(wl,srf_wl,srf[band].to_numpy(float),left=0,right=0)
            good=np.isfinite(reflectance)&np.isfinite(weight)
            denominator=trapezoid(weight[good],wl[good])
            row['Opt_'+band]=(float(trapezoid(reflectance[good]*weight[good],wl[good])
                                      /denominator) if denominator>0 else np.nan)
        row['Opt_NDVI']=index_ratio(row['Opt_B8'],row['Opt_B4'])
        row['Opt_GNDVI']=index_ratio(row['Opt_B8'],row['Opt_B3'])
        row['Opt_NDRE']=index_ratio(row['Opt_B8A'],row['Opt_B5'])
        opt_rows.append(row)

opt=pd.DataFrame(opt_rows)
qc=pd.DataFrame(qc_rows)
assert len(opt)==147 and opt.sample_id.nunique()==147,(len(opt),opt.sample_id.nunique())
opt.to_csv(OUT/'01_OPTOSKY_147_RECOMPUTED_FROM_TXT.csv',index=False,encoding='utf-8-sig')
qc.to_csv(OUT/'02_OPTOSKY_ALL_155_QC.csv',index=False,encoding='utf-8-sig')

# ================================================================
# 3. RECREATE 149 GEE POINT TABLE, JOIN BY EXACT GLOBALID
# ================================================================
gee=pd.read_csv(io.BytesIO(gee_bytes),dtype={'original_GIS_label':str,
                  'paired_label_confirmed':str,'globalid':str})
# Map the point-only Sentinel-2B export to stable internal column names.
# No 3x3 variables are present or used.
gee=gee.rename(columns={'GEE_system_index':'system:index',
    'GEE_snapped_geometry':'.geo','gps_longitude':'original_lon',
    'gps_latitude':'original_lat','original_GIS_label':'nuqta_raqa',
    'paired_label_confirmed':'final_sample_id',
    **{'Sentinel2B_'+b:b for b in BANDS},
    **{'Sentinel2B_'+ix:ix for ix in INDEXES}})

assert len(gee)==149 and gee.globalid.map(norm_gid).nunique()==149
gee['gid']=gee.globalid.map(norm_gid)
for b in BANDS:
    gee[b]=pd.to_numeric(gee[b],errors='raise')
for index in INDEXES: gee[index]=pd.to_numeric(gee[index],errors='raise')
gee['SAT_NDVI_calc']=(gee.B8-gee.B4)/(gee.B8+gee.B4)
gee['SAT_GNDVI_calc']=(gee.B8-gee.B3)/(gee.B8+gee.B3)
gee['SAT_NDRE_calc']=(gee.B8A-gee.B5)/(gee.B8A+gee.B5)
for ix in INDEXES:
    assert np.allclose(gee[ix],gee['SAT_'+ix+'_calc'],atol=2e-6)
gee[['globalid','nuqta_raqa','final_sample_id','original_lon','original_lat',
     *BANDS,*INDEXES]].to_csv(OUT/'03_GEE_149_POINT_VALUES.csv',index=False,encoding='utf-8-sig')

cw=pd.read_csv(io.BytesIO(crosswalk_bytes),dtype={'sample_id':str,'globalid_norm':str})
cw['gid']=cw.globalid_norm.map(norm_gid)
assert len(cw)==144 and cw.gid.nunique()==144 and cw.sample_id.nunique()==144
# NO mean/median of duplicated field labels here. Use confirmed unique GlobalID.
paired=(cw[['sample_id','gid']]
        .merge(gee[['gid','globalid','original_lon','original_lat','.geo',
                   *BANDS,*INDEXES]],on='gid',validate='one_to_one')
        .merge(opt,on='sample_id',validate='one_to_one'))
assert len(paired)==144 and (paired.role=='wheat').sum()==140 and (paired.role=='soil').sum()==4
assert paired.sample_id.ne('2.10').all()
assert {'8.6','8.7'}.issubset(set(paired.sample_id))
paired=paired.sort_values('sample_id',key=lambda x:x.map(lambda t:tuple(int(v) for v in t.split('.'))))
paired=paired.reset_index(drop=True)
for ix in INDEXES:
    paired[f'OPT_minus_S2_{ix}']=paired['Opt_'+ix]-paired[ix]
paired.to_csv(OUT/'04_FINAL_PAIRED_144_FROM_RAW.csv',index=False,encoding='utf-8-sig')
pd.concat([paired.assign(index=ix,Sentinel_value=paired[ix],
                         Optosky_value=paired['Opt_'+ix])
           [['gid','sample_id','role','index','Sentinel_value','Optosky_value']]
           for ix in INDEXES],ignore_index=True).to_csv(
           OUT/'05_PAIRED_LONG_432.csv',index=False,encoding='utf-8-sig')

# Non-calculation verification against archived reported paired results.
ref=pd.read_csv(io.BytesIO(reference_bytes),dtype={'sample_id_TEXT':str})
ref['gid']=ref.globalid.map(norm_gid)
rr=paired.merge(ref,on='gid',validate='one_to_one',suffixes=('','_reference'))
verify=[]
for ix in INDEXES:
    for sensor,candidate,expected in [('S2B',ix,f'Sentinel2B_{ix}'),
                                      ('Optosky','Opt_'+ix,f'Optosky_{ix}')]:
        error=np.abs(rr[candidate].astype(float)-rr[expected].astype(float))
        verify.append(dict(sensor=sensor,index=ix,max_abs_difference=error.max(),
                           n_compared=len(error)))
assert max(v['max_abs_difference'] for v in verify)<1e-9
pd.DataFrame(verify).to_csv(OUT/'06_INDEPENDENT_REFERENCE_CHECK.csv',index=False)

# ================================================================
# 4. RECONSTRUCT PIXEL GROUPS FROM GEE EXPORT CENTERS (NOT NATIVE20 PROOF)
# ================================================================
proj=Transformer.from_crs('EPSG:4326','EPSG:32642',always_xy=True)
def point_from_geo(v): return json.loads(v)['coordinates']
co=np.asarray([point_from_geo(v) for v in gee['.geo']],float)
gee['center_e'],gee['center_n']=proj.transform(co[:,0],co[:,1])
# Original GEE export geometry: centers of 10m sampling grid
assert np.max(np.abs(np.mod(gee.center_e,10)-5))<0.001
assert np.max(np.abs(np.mod(gee.center_n,10)-5))<0.001
gee['pixel10_id']=((np.floor(gee.center_e/10).astype(int)).astype(str)+':'+
                   (np.floor(gee.center_n/10).astype(int)).astype(str))
phases=[]
for eo in (0,10):
    for no in (0,10):
        px=np.floor((gee.center_e-eo)/20).astype(int)
        py=np.floor((gee.center_n-no)/20).astype(int)
        key=px.astype(str)+':'+py.astype(str)
        data=gee.assign(temp_pixel20=key)
        diffs=data.groupby('temp_pixel20')[['B5','B8A']].agg(lambda v:v.max()-v.min())
        phases.append(dict(east_offset=eo,north_offset=no,
                conflicts=int((diffs.max(axis=1)>1e-9).sum()),n_cells=len(diffs)))
phase=pd.DataFrame(phases)
phase.to_csv(OUT/'07_20M_PHASE_SENSITIVITY_NOT_NATIVE_VERIFICATION.csv',index=False)
assert (phase.conflicts==0).sum()==1, '20m inferred phase ambiguous: native GEE export needed'
winner=phase.loc[phase.conflicts.idxmin()]
gee['pixel20_id']=((np.floor((gee.center_e-winner.east_offset)/20).astype(int)).astype(str)
                   +':'+(np.floor((gee.center_n-winner.north_offset)/20).astype(int)).astype(str))
for pcol,bands in [('pixel10_id',['B3','B4','B8']),('pixel20_id',['B5','B8A'])]:
    for b in bands:
        assert gee.groupby(pcol)[b].agg(lambda z:z.max()-z.min()).max()<1e-9
paired=paired.merge(gee[['gid','center_e','center_n','pixel10_id','pixel20_id']],on='gid',
                    validate='one_to_one')
paired[['sample_id','role','gid','pixel10_id','pixel20_id']].to_csv(
                    OUT/'08_RECONSTRUCTED_PIXELS_144.csv',index=False)
occup=[]
for group,d in [('all149',gee),('full_cover144',paired),('wheat140',paired[paired.role=='wheat'])]:
    for resolution in ('pixel10_id','pixel20_id'):
        fr=d[resolution].value_counts()
        occup.append(dict(group=group,grid=resolution,n_rows=len(d),n_pixels=len(fr),
                  extra_rows_in_shared_pixels=int(len(d)-len(fr)),
                  shared_pixels=int((fr>1).sum()),rows_in_shared_pixels=int(fr[fr>1].sum())))
pd.DataFrame(occup).to_csv(OUT/'09_PIXEL_OCCUPANCY.csv',index=False)

# ================================================================
# 5. ASSOCIATION + AGREEMENT; R2=PEARSON-r-SQUARED (NOT PREDICTION R2)
# ================================================================
def agreement(x,y):  # x Sentinel, y Optosky
    x=np.asarray(x,float);y=np.asarray(y,float);m=np.isfinite(x)&np.isfinite(y)
    x=x[m];y=y[m];n=len(x)
    if n<4 or x.std()==0 or y.std()==0:return {'n':n}
    r=float(pearsonr(x,y).statistic);rho=float(spearmanr(x,y).statistic)
    d=y-x;cv=np.mean((x-x.mean())*(y-y.mean()))
    slope=float(cv/np.var(x));intercept=float(y.mean()-slope*x.mean())
    return dict(n=n,Pearson_r=r,Pearson_r2=r*r,Spearman_rho=rho,
       RMSE=float(np.sqrt(np.mean(d*d))),MAE=float(np.mean(abs(d))),
       CCC=float(2*cv/(np.var(x)+np.var(y)+(x.mean()-y.mean())**2)),
       bias=float(d.mean()),LoA_lower=float(d.mean()-1.96*d.std(ddof=1)),
       LoA_upper=float(d.mean()+1.96*d.std(ddof=1)),
       OLS_slope=slope,OLS_intercept=intercept)

stats=[];pixels={}
for pop,d in [('full_cover',paired),('wheat_only',paired[paired.role=='wheat'])]:
    for ix in INDEXES:
        grid=('pixel20_id' if ix=='NDRE' else 'pixel10_id')
        stats.append(dict(population=pop,index=ix,method='field_pairs',grid=grid,
                          **agreement(d[ix],d['Opt_'+ix])))
        med=(d.groupby(grid,sort=False).agg(sat=(ix,'median'),opt=('Opt_'+ix,'median'),
               east=('center_e','median'),north=('center_n','median'),
               n_spectra=('sample_id','size')).reset_index())
        pixels[(pop,ix)]=med
        med.to_csv(OUT/f'10_PIXELS_{pop}_{ix}.csv',index=False)
        stats.append(dict(population=pop,index=ix,method='pixel_median_sensitivity',
                          grid=grid,**agreement(med.sat,med.opt)))
        first=d.drop_duplicates(grid,keep='first')
        stats.append(dict(population=pop,index=ix,method='one_first_spectrum_per_pixel',
                          grid=grid,**agreement(first[ix],first['Opt_'+ix])))
stats=pd.DataFrame(stats)
stats.to_csv(OUT/'11_FINAL_STATS_ALL_INDEXES.csv',index=False,encoding='utf-8-sig')

# ================================================================
# 6. QC / SOIL / DUPLICATE LABEL SENSITIVITY
# ================================================================
sensitivity=[]
def add_scenario(name,data):
    for pop,dd in [('full_cover',data),('wheat_only',data[data.role=='wheat'])]:
        for ix in INDEXES:
            sensitivity.append(dict(scenario=name,population=pop,index=ix,
                                    **agreement(dd[ix],dd['Opt_'+ix])))
add_scenario('primary144',paired)
add_scenario('remove_QC_excess_gt1p5',paired[paired.QC_above1p5==0])
add_scenario('remove_2p7_8p5_10p8',paired[~paired.sample_id.isin(['2.7','8.5','10.8'])])
for s in sorted(SOIL_IDS):add_scenario('leave_out_soil_'+s,paired[paired.sample_id!=s])
add_scenario('wheat_only',paired[paired.role=='wheat'])

# Restore 2.10 ONLY for explicit influence/sensitivity, not primary.
sample_210=opt[opt.sample_id=='2.10'];sat_210=gee[gee.final_sample_id=='2.10']
assert len(sample_210)==1 and len(sat_210)==1
# Restore the authentic 2.10 metadata as well as its spectra and indices.
# Previously only the six index columns were replaced, which had no effect on
# the reported scalar sensitivity metrics but left unrelated GPS/GID values.
extra=paired.iloc[[0]].copy()
source_sat=sat_210.iloc[0]
source_opt=sample_210.iloc[0]
for column in extra.columns:
    if column in sat_210.columns:
        extra.loc[:,column]=source_sat[column]
    if column in sample_210.columns:
        extra.loc[:,column]=source_opt[column]
extra.loc[:,'sample_id']='2.10'
extra.loc[:,'role']='wheat'
extra.loc[:,'gid']=source_sat['gid']
extra.loc[:,'globalid']=source_sat['globalid']
extra.loc[:,'center_e']=source_sat['center_e']
extra.loc[:,'center_n']=source_sat['center_n']
extra.loc[:,'pixel10_id']=source_sat['pixel10_id']
extra.loc[:,'pixel20_id']=source_sat['pixel20_id']
for ix in INDEXES:
    extra.loc[:,'OPT_minus_S2_'+ix]=float(source_opt['Opt_'+ix]-source_sat[ix])
assert extra.iloc[0]['globalid']==source_sat['globalid']
assert extra.iloc[0]['source_file']==source_opt['source_file']
pre145=pd.concat([paired,extra],ignore_index=True)
add_scenario('preQC_145_including_2p10',pre145)
pd.DataFrame(sensitivity).to_csv(OUT/'12_QC_SOIL_AND_2p10_SENSITIVITY.csv',index=False)

# Cook D: OLS response Optosky; predictor Sentinel (same as manuscript)
def cooks(x,y):
    x=np.asarray(x,float);y=np.asarray(y,float);n=len(x)
    X=np.c_[np.ones(n),x]
    beta=np.linalg.lstsq(X,y,rcond=None)[0]
    residual=y-X@beta
    H=np.sum(X*(X@np.linalg.inv(X.T@X)),axis=1)
    mse=np.sum(residual**2)/(n-2)
    return (residual**2/(2*mse))*H/(1-H)**2
cook_rows=[]
for stage,d in [('before_2p10_exclusion145',pre145),('primary144',paired)]:
    for ix in INDEXES:
        di=cooks(d[ix],d['Opt_'+ix]);threshold=4/len(d)
        for sid,role,val in zip(d.sample_id,d.role,di):
            cook_rows.append(dict(stage=stage,index=ix,sample_id=sid,cover=role,
                                  CookD=val,threshold_4n=threshold,flag=val>threshold))
cook_df=pd.DataFrame(cook_rows)
cook_df.to_csv(OUT/'13_COOK_DISTANCE.csv',index=False)

# ================================================================
# 7. INDEPENDENCE SENSITIVITY: BOOTSTRAP + SPATIAL MORAN
# ================================================================
def bootstrap_pixel(med,method,seed,rep=600,block=100):
    rng=np.random.default_rng(seed)
    if method=='pixel': groups=[np.array([i]) for i in range(len(med))]
    else:
        keys=(np.floor(med.east.to_numpy()/block).astype(int).astype(str)+':' +
              np.floor(med.north.to_numpy()/block).astype(int).astype(str))
        codes,unique=pd.factorize(keys)
        groups=[np.where(codes==i)[0] for i in range(len(unique))]
    if len(groups)<8:return dict(valid=0)
    result=[]
    for _ in range(rep):
        chosen=rng.integers(0,len(groups),size=len(groups))
        ids=np.concatenate([groups[j] for j in chosen])
        q=agreement(med.sat.iloc[ids],med.opt.iloc[ids])
        if np.isfinite(q.get('Pearson_r2',np.nan)):
            result.append([q['Pearson_r2'],q['CCC'],q['RMSE']])
    if not result:return dict(valid=0)
    ar=np.array(result)
    ans={'valid':len(result),'n_blocks':len(groups)}
    for j,k in enumerate(('R2','CCC','RMSE')):
        ans[k+'_lower95'],ans[k+'_upper95']=np.percentile(ar[:,j],[2.5,97.5])
    return ans

def moran_absolute_error(med,seed,permutations=999,radius=60,k=8):
    xy=med[['east','north']].to_numpy(float)
    z=np.abs(med.opt.to_numpy(float)-med.sat.to_numpy(float));n=len(z)
    dist,neighbors=cKDTree(xy).query(xy,k=min(k+1,n))
    W=np.zeros((n,n),float)
    for i in range(n):
        for j,d in zip(np.atleast_1d(neighbors[i]),np.atleast_1d(dist[i])):
            if j!=i and d<=radius:W[i,int(j)]=1/max(float(d),0.3)
    W=(W+W.T)/2
    rs=W.sum(axis=1)
    W=np.divide(W,rs[:,None],out=np.zeros_like(W),where=rs[:,None]>0)
    s0=W.sum();v=z-z.mean()
    if s0==0 or v@v==0:return dict(Moran_I=np.nan,permutation_p=np.nan)
    wi, wj = np.nonzero(W)
    weights = W[wi, wj]
    def calc(a):
        return float(n / s0 * np.sum(weights * a[wi] * a[wj]) / np.dot(a, a))
    obs=calc(v)
    rng=np.random.default_rng(seed)
    draws=np.array([calc(rng.permutation(v)) for _ in range(permutations)])
    return dict(Moran_I=obs,permutation_p=(1+np.count_nonzero(abs(draws)>=abs(obs)))
                                   /(permutations+1),neighbour_links=int(np.count_nonzero(W)))

ci_rows=[];moran_rows=[]
for i,((pop,ix),med) in enumerate(pixels.items()):
    for method,block in [('pixel',0),('block100m',100),('block200m',200)]:
        ci_rows.append(dict(population=pop,index=ix,method=method,
                            **bootstrap_pixel(med,method=method,block=block,
                                   rep=BOOTSTRAP_REPEATS,seed=RANDOM_SEED+i*5+block)))
    moran_rows.append(dict(population=pop,index=ix,n_pixels=len(med),
                      **moran_absolute_error(med,RANDOM_SEED+i,
                                          permutations=MORAN_PERMUTATIONS)))
pd.DataFrame(ci_rows).to_csv(OUT/'14_BOOTSTRAP_PIXEL_BLOCK_SENSITIVITY.csv',index=False)
pd.DataFrame(moran_rows).to_csv(OUT/'15_MORAN_ABSOLUTE_ERROR.csv',index=False)

# ================================================================
# 8. ALL STATISTICAL FIGURES: SCATTER, BA, COOK, DISTRIBUTIONS, SRF
# ================================================================
def finish(fig,path):
    fig.tight_layout();fig.savefig(OUT/path,dpi=FIGURE_DPI,bbox_inches='tight')
    plt.close(fig)

for pop,d in [('full_cover',paired),('wheat_only',paired[paired.role=='wheat'])]:
    for ix in INDEXES:
        for method,table,xname,yname in [
            ('field',d,ix,'Opt_'+ix),
            ('pixel_median',pixels[(pop,ix)],'sat','opt')]:
            x=table[xname].to_numpy(float);y=table[yname].to_numpy(float)
            st=agreement(x,y)
            fig,ax=plt.subplots(figsize=(6,5.3));ax.scatter(x,y,s=26,alpha=.75)
            low=min(x.min(),y.min())-.01;high=max(x.max(),y.max())+.01
            ax.plot([low,high],[low,high],'--',lw=1,label='1:1')
            ax.plot([low,high],st['OLS_intercept']+st['OLS_slope']*np.array([low,high]),
                    lw=1,label='OLS')
            ax.set(xlabel=f'Sentinel-2B {ix}',ylabel=f'Optosky SRF {ix}')
            ax.text(.03,.97,f'n={st["n"]}  r²={st["Pearson_r2"]:.3f}\n'
                  f'CCC={st["CCC"]:.3f}  RMSE={st["RMSE"]:.3f}',
                  transform=ax.transAxes,va='top',fontsize=9,
                  bbox=dict(boxstyle='round',facecolor='white',alpha=.8))
            ax.legend(loc='lower right');finish(fig,f'FIG_SCATTER_{pop}_{ix}_{method}.png')
        x=d[ix].to_numpy(float);y=d['Opt_'+ix].to_numpy(float)
        difference=y-x;bias=difference.mean();std=difference.std(ddof=1)
        fig,ax=plt.subplots(figsize=(6,4.8))
        ax.scatter((x+y)/2,difference,s=24,alpha=.7)
        ax.axhline(bias,label='Mean bias')
        ax.axhline(bias+1.96*std,ls='--',label='±1.96 SD')
        ax.axhline(bias-1.96*std,ls='--')
        ax.set(xlabel=f'Mean of Optosky and Sentinel-2B {ix}',
               ylabel='Optosky − Sentinel-2B')
        ax.legend();finish(fig,f'FIG_BLAND_ALTMAN_{pop}_{ix}.png')

for ix in INDEXES:
    d=cook_df[(cook_df.stage=='before_2p10_exclusion145')&(cook_df['index']==ix)]
    d=d.sort_values('CookD',ascending=False).head(15)
    fig,ax=plt.subplots(figsize=(8,4.5));ax.bar(d.sample_id,d.CookD)
    ax.axhline(4/145,ls='--',label='4/n')
    ax.set(xlabel='Spectrum ID',ylabel="Cook's distance")
    ax.legend();finish(fig,f'FIG_COOK_PRE145_{ix}.png')

# Actual instrument spectra and official SRF curves
fig,ax=plt.subplots(figsize=(8.5,5))
for sample in ('2.10','3.7','3.9','5.5'):
    if sample in saved_curves:
        wl,refc=saved_curves[sample];sel=(wl>=450)&(wl<=900)&np.isfinite(refc)
        ax.plot(wl[sel],refc[sel],lw=.9,label=sample)
ax.set(xlabel='Wavelength (nm)',ylabel='Optosky C4/C3');ax.legend()
finish(fig,'FIG_REAL_SPECTRA_QC.png')
fig,ax=plt.subplots(figsize=(8.5,5))
for b in BANDS:
    ax.plot(srf.wavelength_nm,srf[b],lw=1.1,label=b)
ax.set_xlim(450,950);ax.set(xlabel='Wavelength (nm)',
             ylabel='Relative spectral response')
ax.legend();finish(fig,'FIG_OFFICIAL_S2B_SRF.png')

fig,axes=plt.subplots(2,3,figsize=(12,7))
for i,ix in enumerate(INDEXES):
    for j,(pop,d) in enumerate([('All cover',paired),('Wheat only',paired[paired.role=='wheat'])]):
        ax=axes[j,i]
        ax.hist(d[ix],bins=18,alpha=.55,label='Sentinel-2B')
        ax.hist(d['Opt_'+ix],bins=18,alpha=.5,label='Optosky')
        ax.set(xlabel='Index value',ylabel='Observations')
        if i==0 and j==0:ax.legend()
finish(fig,'FIG_INDEX_HISTOGRAMS_6PANELS.png')

fig,axes=plt.subplots(2,3,figsize=(12,7))
for i,ix in enumerate(INDEXES):
    for j,(pop,d) in enumerate([('Full cover',paired),('Wheat only',paired[paired.role=='wheat'])]):
        ax=axes[j,i];x=d[ix].to_numpy(float);y=d['Opt_'+ix].to_numpy(float)
        ax.scatter(x,y,s=12,alpha=.6)
        lo=min(x.min(),y.min());hi=max(x.max(),y.max())
        ax.plot([lo,hi],[lo,hi],'--',lw=1)
        ax.set(xlabel=f'Sentinel {ix}',ylabel=f'Optosky {ix}',)
finish(fig,'FIG_ALL_6_SCATTERS.png')

# ================================================================
# 9. README + SAVE ALL OUTPUT FILES AS ONE ZIP
# ================================================================
readme=f'''BO'KA single-file reproducible audit (2026-11-09)
Input package: {package.name}
Raw spectral TXT: {len(txt_names)}; convolved eligible: {len(opt)}
GEE source records: {len(gee)}; valid GlobalID pairs: {len(paired)};
Wheat: {(paired.role=='wheat').sum()}; soil: {(paired.role=='soil').sum()}

Inputs from raw 155 TXT, official S2B SRF and original GEE export 149.
GlobalID crosswalk is an author-confirmed external audit decision, not an
algorithmic inference. 2.10 excluded from primary due to QC (sensitivity shown).

Pearson_r2 is squared Pearson association, NOT predictive cross-validated R2.
CCC, RMSE, MAE and Bland-Altman assess agreement separately.

(OUT/'README_METHODS_AND_LIMITATIONS.txt').write_text(readme,encoding='utf-8')

print('\n================ REPRODUCED RESULTS ================')
print(stats[(stats.method=='field_pairs')][['population','index','n','Pearson_r2','CCC','RMSE']]
      .to_string(index=False,float_format=lambda v:f'{v:.6f}'))
print('\nPixel occupancy:')
print(pd.DataFrame(occup).to_string(index=False))
print('\nQC-flagged spectra:')
print(qc.query('above_1p5>0')[['sample_id','above_1p5']].to_string(index=False))

# Publication figures: the descriptive captions are stored in the manuscript.
"""Publication figures for the Boka 2026 spectral comparison.
All plotted measurements originate in the primary TXT spectra and the GEE point export.
Figure titles are intentionally omitted; descriptive captions are kept in the manuscript.
"""
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch
from pathlib import Path
from scipy.stats import pearsonr
from scipy.integrate import trapezoid

# These are assigned by the main pipeline on execution:
# OUT, paired, gee, saved_curves, srf, cook_df, sensitivity, INDEXES, agreement
FIGURE_DPI=600
PUBLISH=OUT/'JARS_PUBLICATION_FIGURES'
PUBLISH.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'font.size':10,'axes.labelsize':11,'legend.fontsize':9,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'savefig.facecolor':'white','figure.facecolor':'white',
                     'font.family':FIGURE_FONT_FAMILY})
def publish(fig,slug):
    fig.savefig(PUBLISH/f'{slug}.png', dpi=FIGURE_DPI, bbox_inches='tight',
                facecolor='white',pad_inches=0.12)
    fig.savefig(PUBLISH/f'{slug}.pdf',bbox_inches='tight',pad_inches=0.12)
    plt.close(fig)

# Fig. 2 -- actual field spectra, with panel/sample labels only.
fig,ax=plt.subplots(figsize=(7.3,4.2))
for sample,style,label in [('7.7','-', 'Wheat (7.7)'),('3.23','-', 'High-influence wheat (3.23)'),
                           ('2.10','--','QC-excluded sample (2.10)')]:
    if sample not in saved_curves: continue
    wl,r=saved_curves[sample];valid=(wl>=450)&(wl<=900)
    ax.plot(wl[valid],r[valid],ls=style,lw=1.5,label=label)
ax.set(xlabel='Wavelength (nm)',ylabel='C4/C3 spectral ratio',xlim=(450,900))
ax.axvspan(620,690,color='0.7',alpha=.12,lw=0);ax.axvspan(700,740,color='0.5',alpha=.10,lw=0)
ax.legend(frameon=False)
publish(fig,'FIG_02_FIELD_SPECTRAL_QC')

# Fig. 3 -- measured field spectrum with official SRFs (dual y-axis).
fig,ax=plt.subplots(figsize=(7.3,4.2));ax2=ax.twinx()
wl,r=saved_curves['3.23'];valid=(wl>=450)&(wl<=900)
line=ax.plot(wl[valid],r[valid],color='0.18',lw=1.9,label='Optosky spectrum (3.23)')[0]
other=[]
for band in ('B3','B4','B5','B8','B8A'):
    ls=ax2.plot(srf.wavelength_nm,srf[band],lw=1.35,label=band)[0];other.append(ls)
ax.set(xlabel='Wavelength (nm)',ylabel='Field spectral ratio (C4/C3)',xlim=(450,940))
ax2.set_ylabel('Relative spectral response');ax2.spines['right'].set_visible(True)
ax2.set_ylim(0,1.12)
ax.legend([line,*other],[z.get_label() for z in [line,*other]],loc='upper left',ncol=3,
          fontsize=8,frameon=False)
publish(fig,'FIG_03_FIELD_SPECTRUM_AND_SENTINEL_SRF')

# Fig. 4 -- spatial sampling diagram without an internal title.
fig,ax=plt.subplots(figsize=(8.0,3.2));ax.set_xlim(0,12);ax.set_ylim(0,4.3);ax.axis('off')
for i in range(4):
    for j in range(3):
        ax.add_patch(Rectangle((.5+i*.85,.55+j*.85),.85,.85,
                      fc='#eaf0f5',ec='#7890a1',lw=.8))
for x,y in [(1.16,2.4),(1.88,1.43),(2.88,2.62),(3.33,.9)]:
    ax.plot(x,y,marker='o',color='#ba4a36',ms=5)
ax.text(2.15,.20,'10 m sampling grid',ha='center',fontsize=10)
ax.annotate('',xy=(6.1,1.83),xytext=(4.32,1.83),arrowprops=dict(arrowstyle='->',lw=1.6))
ax.add_patch(FancyBboxPatch((6.12,.94),5.0,1.75,boxstyle='round,pad=.12',lw=1,
                            edgecolor='#668577',facecolor='#f1f7f3'))
ax.text(8.62,2.28,'GlobalID-based point matching',ha='center',fontsize=11)
ax.text(8.62,1.76,'Sentinel-2B index at sampled location',ha='center',fontsize=9.3)
ax.text(8.62,1.28,'10 m: NDVI / GNDVI     20 m: NDRE',ha='center',fontsize=9.3)
publish(fig,'FIG_04_POINT_PIXEL_SCHEMATIC')

# Fig. 8 -- processing workflow without internal title or Uzbek labels.
fig,ax=plt.subplots(figsize=(10.0,3.3));ax.set_xlim(0,14);ax.set_ylim(0,4);ax.axis('off')
boxes=[('Field TXT spectra\n155 files',0.25),('Spectral QC\n147 candidates',2.70),
       ('C4/C3 + S2B SRF\n5 MSI bands',5.15),('Index pairs\nNDVI, GNDVI, NDRE',7.60),
       ('GlobalID matching\n144 pairs',10.05)]
for i,(label,x) in enumerate(boxes):
    ax.add_patch(FancyBboxPatch((x,2.10),2.18,1.04,
          boxstyle='round,pad=.10',edgecolor='#6d8599',facecolor='#eff3f7',lw=1))
    ax.text(x+1.09,2.62,label,ha='center',va='center',fontsize=8.5)
    if i:
        ax.annotate('',xy=(x-.10,2.63),xytext=(x-.28,2.63),
                    arrowprops=dict(arrowstyle='->',color='0.28',lw=1.3))
ax.annotate('',xy=(11.13,1.78),xytext=(11.13,2.06),arrowprops=dict(arrowstyle='->',lw=1.4))
ax.add_patch(FancyBboxPatch((8.8,.30),4.55,1.23,boxstyle='round,pad=.10',
      edgecolor='#b9854d',facecolor='#fbf6ee',lw=1))
ax.text(11.075,.95,'Pearson r / r² · RMSE · CCC · bias\nPixel and QC sensitivity analyses',
        ha='center',va='center',fontsize=9.5)
publish(fig,'FIG_08_ANALYTICAL_WORKFLOW')

# Fig. 9 -- pooled and wheat-only empirical scatter, no narrative headline.
wheat = paired[paired.role == 'wheat']
soil = paired[paired.role == 'soil']
fig,axes=plt.subplots(1,3,figsize=(10.5,3.6),sharex=False,sharey=False)
for ax,ix in zip(axes,INDEXES):
    for kind,label,mark in [('wheat','Wheat (n=140)','o'),('soil','Soil (n=4)','s')]:
        part=paired[paired.role==kind]
        ax.scatter(part[ix],part['Opt_'+ix],s=12 if kind=='wheat' else 43,
                   marker=mark,alpha=.8,label=label)
    low=float(min(paired[ix].min(),paired['Opt_'+ix].min()))-.015
    high=float(max(paired[ix].max(),paired['Opt_'+ix].max()))+.015
    ax.plot([low,high],[low,high],color='0.45',lw=.9,ls='--')
    ax.set(xlim=(low,high),ylim=(low,high),xlabel=f'Sentinel-2B {ix}',ylabel=f'Optosky {ix}')
    rp=pearsonr(paired[ix],paired['Opt_'+ix]).statistic**2
    rw=pearsonr(wheat[ix],wheat['Opt_'+ix]).statistic**2
    ax.text(.04,.95,ix,transform=ax.transAxes,va='top',fontweight='bold')
    ax.text(.04,.82,f'Pooled r² = {rp:.3f}\nWheat r² = {rw:.3f}',
            transform=ax.transAxes,va='top',fontsize=8)
axes[0].legend(frameon=False,loc='lower right',fontsize=8)
fig.tight_layout(w_pad=1.25);publish(fig,'FIG_09_POOLED_AND_WHEAT_SCATTER')

# Fig. 10 -- three Bland–Altman panels, only index panel tags.
fig,axes=plt.subplots(1,3,figsize=(10.5,3.6))
for ax,ix in zip(axes,INDEXES):
    x=wheat[ix].to_numpy(float);y=wheat['Opt_'+ix].to_numpy(float)
    center=(x+y)/2;err=y-x;bias=err.mean();sd=err.std(ddof=1)
    ax.scatter(center,err,s=13,alpha=.75)
    ax.axhline(bias,color='0.24',lw=1.2)
    for lim in [bias-1.96*sd,bias+1.96*sd]: ax.axhline(lim,color='0.5',ls='--',lw=1)
    ax.set(xlabel=f'Mean {ix}',ylabel='Optosky − Sentinel-2B')
    ax.text(.04,.96,ix,transform=ax.transAxes,va='top',fontweight='bold')
    ax.text(.04,.86,f'Bias = {bias:+.3f}',transform=ax.transAxes,va='top',fontsize=8)
fig.tight_layout(w_pad=1.3);publish(fig,'FIG_10_WHEAT_BLAND_ALTMAN')

# Fig. 11 -- range constriction in 2-row index distributions.
fig,axes=plt.subplots(2,3,figsize=(10.3,5.4))
for j,ix in enumerate(INDEXES):
    for i,(part,label) in enumerate([(paired,'Pooled'),(wheat,'Wheat only')]):
        ax=axes[i,j]
        ax.hist(part[ix],bins=18,color='#64879b',alpha=.83,edgecolor='white',linewidth=.25)
        ax.set(xlabel=f'Sentinel-2B {ix}',ylabel='Count')
        ax.text(.05,.95,f'{ix} · {label}',transform=ax.transAxes,va='top',fontsize=9)
fig.tight_layout(h_pad=1.4,w_pad=1.2);publish(fig,'FIG_11_INDEX_DISTRIBUTIONS')

# Fig. 12 -- leave-one-soil-out, no figure title.
fig,axes=plt.subplots(1,3,figsize=(10,3.3),sharey=True)
for ax,ix in zip(axes,INDEXES):
    labels=['All soil','−4.1','−4.2','−4.3','−4.4','Wheat only'];rows=[]
    for label in labels:
        q=paired if label=='All soil' else wheat if label=='Wheat only' else paired[paired.sample_id!=label[1:]]
        rows.append(pearsonr(q[ix],q['Opt_'+ix]).statistic**2)
    ax.plot(range(len(rows)),rows,marker='o',lw=1.3,markersize=4)
    ax.set(xticks=range(len(rows)),ylabel='Pearson r²',ylim=(0,1.0))
    ax.set_xticklabels(labels,rotation=38,ha='right',fontsize=7.5)
    ax.text(.06,.92,ix,transform=ax.transAxes,fontweight='bold')
fig.tight_layout(w_pad=1.4);publish(fig,'FIG_12_SOIL_REFERENCE_SENSITIVITY')

# A machine-readable map between figure numbers and publication files.
files={
 '2':'FIG_02_FIELD_SPECTRAL_QC.png',
 '3':'FIG_03_FIELD_SPECTRUM_AND_SENTINEL_SRF.png','4':'FIG_04_POINT_PIXEL_SCHEMATIC.png',
 '8':'FIG_08_ANALYTICAL_WORKFLOW.png','9':'FIG_09_POOLED_AND_WHEAT_SCATTER.png',
 '10':'FIG_10_WHEAT_BLAND_ALTMAN.png','11':'FIG_11_INDEX_DISTRIBUTIONS.png',
 '12':'FIG_12_SOIL_REFERENCE_SENSITIVITY.png'}
pd.DataFrame([{'figure':k,'publication_file':v,'dpi':FIGURE_DPI} for k,v in files.items()]).to_csv(
            PUBLISH/'FIGURE_MANIFEST.csv',index=False)
print('Publication figures prepared:',len(files),'at',FIGURE_DPI,'dpi')

# ================================================================
# 10. ORGANIZE EVERY ARTICLE GRAPH, SUPPLEMENTARY GRAPH, AND TABLE
# ================================================================
import shutil
article_dir=OUT/'01_ARTICLE_PYTHON_FIGURES'
extra_dir=OUT/'02_SUPPLEMENTARY_PYTHON_FIGURES'
table_dir=OUT/'03_EXCEL_TABLES_CSV'
workbook_dir=OUT/'04_EXCEL_WORKBOOK_NOT_REQUIRED'
for folder in (article_dir,extra_dir,table_dir,workbook_dir):
    folder.mkdir(parents=True,exist_ok=True)

# Keep all original generated files at their original relative paths, too.
for fig_number,filename in files.items():
    stem=Path(filename).stem
    for extension in ('.png','.pdf'):
        generated=PUBLISH/(stem+extension)
        if not generated.exists():
            raise RuntimeError(f'Missing required article figure: {generated}')
        shutil.copy2(generated,article_dir/generated.name)

for image in sorted(OUT.glob('FIG_*.png')):
    shutil.copy2(image,extra_dir/image.name)
for data_file in sorted(OUT.glob('*.csv')):
    shutil.copy2(data_file,table_dir/data_file.name)
shutil.copy2(PUBLISH/'FIGURE_MANIFEST.csv',table_dir/'ARTICLE_FIGURE_MANIFEST.csv')

# All 21+ CSV analysis tables are Excel-compatible and are copied above.
# Exporting a bulky multi-sheet workbook is intentionally omitted to keep
# Colab computation reliable. Open any CSV in Excel, then Save As .xlsx.
print('All numerical tables exported as Excel-compatible CSV files.')

summary_rows=[
    {'category':'article','file':f'01_ARTICLE_PYTHON_FIGURES/{Path(name).name}',
     'figure_number':key,'dpi':600,'font':FIGURE_FONT_FAMILY}
    for key,name in files.items()
]
summary_rows.extend({'category':'supplement','file':f'02_SUPPLEMENTARY_PYTHON_FIGURES/{p.name}',
                     'figure_number':'','dpi':600,'font':FIGURE_FONT_FAMILY}
                    for p in sorted(extra_dir.glob('*.png')))
summary_rows.extend({'category':'excel_csv','file':f'03_EXCEL_TABLES_CSV/{p.name}',
                     'figure_number':'','dpi':'','font':''}
                    for p in sorted(table_dir.glob('*.csv')))
pd.DataFrame(summary_rows).to_csv(OUT/'00_OUTPUT_FILE_INDEX.csv',index=False)

font_notice=('Times New Roman (installed)' if FIGURE_FONT_FAMILY=='Times New Roman' else
             'Tinos (Times-compatible alternative, NOT genuine Times New Roman)')
(OUT/'00_START_HERE_UZ.txt').write_text(f"""BO‘KA COLAB — BARCHA PYTHON GRAFIKLARI VA JADVALLARI

01_ARTICLE_PYTHON_FIGURES:
  Maqoladagi Figure 2, 3, 4, 8, 9, 10, 11, 12 (PNG 600 DPI + PDF).
  Figure 1 va 5–7 GIS xaritalari bu notebookda chizilmaydi.
  Figure 1b olib tashlangan, bu yerda qayta yaratilmaydi.

02_SUPPLEMENTARY_PYTHON_FIGURES:
  Barcha boshqa Python grafiklari (PNG 600 DPI).

03_EXCEL_TABLES_CSV:
  Barcha statistik hisoblar, QC, pixel, bootstrap va audit CSV jadvallari.
  CSV fayllarni Excel bevosita ochadi.

04_EXCEL_WORKBOOK:
  CSV jadvallarni Excel bevosita ochadi; Excelda Save As .xlsx qiling.

Shrift: {font_notice}
Barcha Python grafiklar: 600 DPI.
Maqola va qo‘shimcha natijalarning ayrimlari bir xil manbadan olingan,
lekin faqat maqolaga kiritilmagan materialni supplementary sifatida yuboring.

Hisoblar: 155 TXT; 149 GEE eksport; 144 asosiy juftlik (140 wheat + 4 soil).
Koddagi QC va SRF taxminlari original muallif ma’lumotlariga asoslangan.
""",encoding='utf-8')

# The user uploads only one ZIP once. One ZIP is automatically downloaded.
archive_path=OUT.parent/'BOKA_ALL_PYTHON_FIGURES_EXCEL_TABLES_600DPI.zip'
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as output:
    for entry in sorted(OUT.rglob('*')):
        if entry.is_file():
            output.write(entry,arcname=str(entry.relative_to(OUT)))
assert len(list(article_dir.glob('*.png')))==8, 'Expected exactly 8 article Python PNG figures'
print('\nDONE:',archive_path)
print('Article Python figures:',len(list(article_dir.glob('*.png'))))
print('Supplementary Python figures:',len(list(extra_dir.glob('*.png'))))
print('Excel-compatible CSV tables:',len(list(table_dir.glob('*.csv'))))
print('Font:',FIGURE_FONT_FAMILY,' / PNG DPI:',FIGURE_DPI)
if Path('/content').is_dir():
    from google.colab import files as colab_files
    colab_files.download(str(archive_path))


UPLOAD THE FINAL DATA PACKAGE ZIP:  BOKA_COLAB_INPUTS_ONLY_20261109.zip


Saving BOKA_COLAB_INPUTS_ONLY_20261109.zip to BOKA_COLAB_INPUTS_ONLY_20261109.zip
FONT NOTICE: Times New Roman is unavailable in this runtime;
Tinos (free metric-compatible Times-style font) will be used automatically.
All graph labels font: Tinos

================ REPRODUCED RESULTS ================
population index   n  Pearson_r2      CCC     RMSE
full_cover  NDVI 144    0.861221 0.908682 0.062861
full_cover GNDVI 144    0.791743 0.863274 0.059877
full_cover  NDRE 144    0.780904 0.841418 0.074759
wheat_only  NDVI 140    0.159133 0.352171 0.060394
wheat_only GNDVI 140    0.197345 0.412498 0.054253
wheat_only  NDRE 140    0.228452 0.392475 0.074427

Pixel occupancy:
        group       grid  n_rows  n_pixels  extra_rows_in_shared_pixels  shared_pixels  rows_in_shared_pixels
       all149 pixel10_id     149       139                           10             10                     20
       all149 pixel20_id     149       105                           44             32                 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>